# 泛型与静态类型检查

学习目标：用泛型和协议表达输入、输出及对象接口的关系，运行静态类型检查，并判断哪些约束仍需运行时校验。

前置知识：类型标注、函数与装饰器、类与继承、迭代器、模块导入及文件读写。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

静态检查使用 mypy 2.3.1。

配套脚本：位于 [scripts/17-static-typing/](scripts/17-static-typing/)。

1. [forward_models.py](scripts/17-static-typing/forward_models.py)：演示延后求值的标注与 TYPE_CHECKING。
2. [report_value.py](scripts/17-static-typing/report_value.py)、[report_value.pyi](scripts/17-static-typing/report_value.pyi)：演示实现文件与类型声明文件。
3. [report_client.py](scripts/17-static-typing/report_client.py)：演示调用方如何使用上述类型声明。

## 1 运行静态类型检查

类型标注基础介绍了单个值的类型；本章进一步表达多个类型之间的关系，并使用 mypy 检查这些约定。

mypy 分析源代码，不执行被检查的程序。终端可用 python -m mypy --strict 文件名.py；下面通过 mypy.api.run() 在 Notebook 中调用同一个检查器。

check_types() 接收一段独立的源码字符串，不共享 Notebook 中的变量。输出中的行号对应这段源码；方括号内是错误代码。errors 用来列出反例应出现的错误，避免把工具故障当成预期失败。

In [1]:
import os
from textwrap import dedent

from mypy import api


def check_types(source: str, *, errors: tuple[str, ...] = ()) -> None:
    """检查独立源码，展示诊断并确认是否出现预期类型错误。"""
    # 1. 指定目标版本和严格检查；不留下类型检查缓存。
    report, error_report, status = api.run([
        "--python-version", "3.12", "--strict",
        "--no-incremental", "--cache-dir", os.devnull,
        "--no-error-summary", "--no-pretty", "--no-color-output",
        "-c", dedent(source).strip(),
    ])

    # 2. 原样显示诊断，只接受约定的错误种类和退出状态。
    print(report.strip() or "未发现类型错误")  # 错误示例显示带方括号错误码的诊断；正确示例显示“未发现类型错误”。
    assert not error_report, error_report
    actual = tuple(
        line.rsplit("[", 1)[1].removesuffix("]")
        for line in report.splitlines() if ": error:" in line
    )
    assert status == (1 if errors else 0), status
    assert actual == errors, actual


# 静态检查报告 assignment；这段字符串中的程序不会执行。
check_types('count: int = "3"', errors=("assignment",))
check_types("count: int = 3")

<string>:1: error: Incompatible types in assignment (expression has type "str", variable has type "int")  [assignment]


未发现类型错误


## 2 泛型函数、类与类型别名

### 2.1 用类型参数保留对应关系

泛型（generic）把某些类型留作参数，使用时再确定。T 是类型参数名，表示本次使用中的某个类型；它不是普通数据变量。

Python 3.12 可以在函数名、类名或 type 别名后写 [T]。例如 first() 的输入元素和返回值都用 T，检查器就能保留两者的对应关系。Sequence 表示支持索引等序列操作的接口。

In [2]:
from collections.abc import Sequence


def first[T](items: Sequence[T]) -> T:
    """返回序列首项；空序列抛出 IndexError。"""
    return items[0]


class Box[T]:
    """保存一个具有指定类型的值。"""

    def __init__(self, value: T) -> None:
        self.value = value


type Pair[T] = tuple[T, T]

number = first([8, 9])
word = first(("Python", "typing"))
box = Box[int](number)
pair: Pair[str] = (word, "mypy")

print(number, word, box.value, pair)  # 8 Python 8 ('Python', 'mypy')
# 泛型没有改变 first 的索引行为；空序列仍然不满足调用条件。

8 Python 8 ('Python', 'mypy')


### 2.2 TypeVar、上界与约束

旧写法通过 TypeVar() 显式创建类型变量，通过 Generic[T] 定义泛型类；这些写法在 Python 3.12 中仍然有效。

T: str 表示类型上界，允许 str 及其子类；T: (str, bytes) 表示有限类型约束，同一次调用中的 T 必须统一为其中一种。不要把后者理解为两个参数各自任选一种联合类型。

In [3]:
from typing import Generic, TypeVar

Value = TypeVar("Value")


def keep(value: Value) -> Value:
    """返回原值，保留输入类型。"""
    return value


class LegacyBox(Generic[Value]):
    """演示使用 Generic 和 TypeVar 定义类。"""

    def __init__(self, value: Value) -> None:
        self.value = value


def keep_text[T: str](value: T) -> T:
    """接收字符串或其子类，并返回原对象。"""
    return value


def join_same[T: (str, bytes)](left: T, right: T) -> T:
    """拼接两个同为字符串或同为字节串的值。"""
    return left + right


print(keep(8), LegacyBox("说明").value)  # 8 说明
print(keep_text("名称"), join_same(b"a", b"b"))  # 名称 b'ab'

8 说明
名称 b'ab'


### 2.3 观察检查器推断的类型

reveal_type() 让检查器报告表达式的静态类型。下面只把它放在送给 mypy 的源码中，不调用它的运行时版本。

泛型函数调用时从实参推断 T，写 first([8])，不用 first\[int\]([8])。泛型类则可以显式写 Box\[int\](8)。

In [4]:
check_types("""
    from collections.abc import Sequence

    def first[T](items: Sequence[T]) -> T:
        return items[0]

    reveal_type(first([8, 9]))
    reveal_type(first(("a", "b")))
""")
# 两条 note 分别显示 int 和 str；没有类型错误。

<string>:6: note: Revealed type is "int"
<string>:7: note: Revealed type is "str"


## 3 容器接口与迭代类型

### 3.1 按需要的操作选择标注

容器类型强调保存什么，迭代类型强调能执行什么操作；同一个列表既可以满足 list[int]，也可以作为 Iterable[int] 使用。

下表中 T 表示元素类型，Y、S、R 分别表示生成器产出值、send() 接收值和最终返回值的类型。

| 名称／写法 | 中文含义／用途 |
| --- | --- |
| Iterable[T] | 可迭代对象，适合只需遍历的输入 |
| Iterator[T] | 迭代器，支持 next()，会逐步耗尽 |
| Sequence[T] | 序列接口，适合需要索引、长度等操作的输入 |
| Generator[Y, S, R] | 生成器，分别描述产出、接收和返回的类型 |

本章从 collections.abc 导入这些接口。只产出值的生成器通常标成 Iterator[T] 就足够。

In [5]:
from collections.abc import Generator, Iterable, Iterator


def nonempty(lines: Iterable[str]) -> Iterator[str]:
    """惰性产出去除两端空白后的非空行。"""
    for line in lines:
        cleaned = line.strip()
        if cleaned:
            yield cleaned


def confirm() -> Generator[str, bool, int]:
    """先产出提示，再接收确认值并返回状态码。"""
    # Generator 的三个类型参数依次对应产出值、send 输入、最终返回值。
    accepted = yield "继续？"
    return 1 if accepted else 0


print(list(nonempty([" a ", "", "b"])))  # ['a', 'b']
dialog = confirm()
print(next(dialog))  # 继续？
try:
    dialog.send(True)
except StopIteration as error:
    print(error.value)  # 1：最终 return，不是一次 yield。
else:
    raise AssertionError("生成器应在确认后结束")

['a', 'b']
继续？
1


### 3.2 可变容器不能随意扩大元素类型

list[str] 不能赋给 list[object]：后者允许追加整数，会破坏原列表“只有字符串”的约定。这称为不变（invariant）。

只需读取时，可使用 Sequence[object] 接收 list[str]。Sequence 的元素类型是协变（covariant）的；这种接口限制可用操作，不会把底层列表变成不可变对象。

In [6]:
check_types("""
    from collections.abc import Sequence

    words: list[str] = ["a"]
    readable: Sequence[object] = words
    writable: list[object] = words
""", errors=("assignment",))
# 错误只针对 writable；readable 不能通过 Sequence 接口执行 append。

<string>:5: error: Incompatible types in assignment (expression has type "list[str]", variable has type "list[object]")  [assignment]
<string>:5: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:5: note: Consider using "Sequence" instead, which is covariant


## 4 Callable 与 ParamSpec

### 4.1 标注可调用对象

Callable[[int], str] 表示接收一个整数实参、返回字符串的可调用对象。方括号中的参数列表描述参数类型，最后一项描述返回类型。

Callable[..., str] 只约定返回字符串，省略号表示不限制参数列表；需要检查调用参数时，不要用它抹去已有信息。

In [7]:
from collections.abc import Callable


def apply_text(value: int, formatter: Callable[[int], str]) -> str:
    """用传入的函数把整数格式化为文本。"""
    return formatter(value)


def format_score(score: int) -> str:
    """生成分数文本。"""
    return f"{score} 分"


print(apply_text(90, format_score))  # 90 分

90 分


### 4.2 让装饰器保留参数列表

ParamSpec 表示一整组函数参数；Python 3.12 写作 \*\*P。P.args 和 P.kwargs 分别用于包装函数的 \*args、\*\*kwargs 标注，R 表示返回类型。

wraps() 负责保留函数元数据，ParamSpec 负责让静态检查器保留参数关系；两者解决的问题不同。旧代码常用 P = ParamSpec("P") 和 R = TypeVar("R") 显式声明。

In [8]:
from functools import wraps


def announce[**P, R](function: Callable[P, R]) -> Callable[P, R]:
    """打印调用提示，并保留被包装函数的参数与返回类型。"""
    @wraps(function)
    def wrapped(*args: P.args, **kwargs: P.kwargs) -> R:
        print("开始调用")
        return function(*args, **kwargs)

    return wrapped


@announce
def title(name: str, *, prefix: str = "课程") -> str:
    """拼接标题，prefix 只能作为关键字实参传入。"""
    return f"{prefix}：{name}"


print(title("Python", prefix="学习"))  # 开始调用，然后是学习：Python

开始调用
学习：Python


### 4.3 检查装饰后的调用

下面检查显式 ParamSpec 写法。即使增加了一层包装，错误的参数类型仍会被发现；声明参数关系不会自动转换实参。

In [9]:
check_types("""
    from collections.abc import Callable
    from typing import ParamSpec, TypeVar

    P = ParamSpec("P")
    R = TypeVar("R")

    def forward(function: Callable[P, R]) -> Callable[P, R]:
        def wrapped(*args: P.args, **kwargs: P.kwargs) -> R:
            return function(*args, **kwargs)
        return wrapped

    @forward
    def label(score: int, *, suffix: str = "分") -> str:
        return f"{score}{suffix}"

    label("90")
""", errors=("arg-type",))
# 检查器仍知道 label 的 score 应为 int。

<string>:16: error: Argument 1 to "label" has incompatible type "str"; expected "int"  [arg-type]


## 5 TypeVarTuple：保留一组元素类型

TypeVarTuple 表示数量可变的一组类型；Python 3.12 用 \*Ts 声明和展开。Ts 是类型参数组的名称，例如它可以对应 str、float 两个类型。

tuple[T, ...] 表示任意长度、元素同型的元组；tuple[\*Ts] 可以保留各位置不同的类型。下面给元组加上字符串标签，同时保留原来的各位置类型。

In [10]:
def tag_values[*Ts](tag: str, values: tuple[*Ts]) -> tuple[str, *Ts]:
    """在元组前加标签，保留原有元素。"""
    return (tag, *values)


print(tag_values("记录", (7, True)))  # ('记录', 7, True)
print(tag_values("空", ()))  # ('空',)

('记录', 7, True)
('空',)


显式构造写作 Ts = TypeVarTuple("Ts")。下面通过检查器确认输出位置的类型；这类写法适合确实需要保留异构元组结构的接口，不必用于普通列表。

In [11]:
check_types("""
    from typing import TypeVarTuple

    Ts = TypeVarTuple("Ts")

    def tag_values(
        tag: str, values: tuple[*Ts]
    ) -> tuple[str, *Ts]:
        return (tag, *values)

    values: tuple[int, bool] = (7, True)
    reveal_type(tag_values("记录", values))
""")
# note 显示 tuple[str, int, bool]。

<string>:11: note: Revealed type is "tuple[str, int, bool]"


## 6 TypedDict：标注字典中的字段

dict[str, int] 表示所有键为字符串、所有值为整数；TypedDict 可以按键名分别指定字段类型，适合结构固定的字典。

| 名称 | 中文含义／用途 |
| --- | --- |
| TypedDict | 描述字典的字段及各字段类型 |
| NotRequired | 允许某个键缺省 |
| Required | 在 total=False 的字典中要求某个键存在 |

默认所有字段必需。键可以缺省与值可以为 None 是两个维度；NotRequired[str] 不等于 str 与 None 的联合类型。运行时创建的对象仍是普通 dict。

In [12]:
from typing import NotRequired, Required, TypedDict


class Result(TypedDict):
    """描述一条分数记录。"""

    name: str
    score: int
    note: NotRequired[str]


class FilterOptions(TypedDict, total=False):
    """查询名必需，最低分可省略。"""

    name: Required[str]
    minimum: int


result: Result = {"name": "小林", "score": 90}
options: FilterOptions = {"name": "小林"}
print(type(result) is dict)  # True
print(result.get("note", "无备注"), options)  # 无备注 {'name': '小林'}

True
无备注 {'name': '小林'}


TypedDict 的字段约束由检查器使用，不会自动检查解析得到的外部数据。输入校验仍要明确检查必需键、字段类型及业务范围。

In [13]:
check_types("""
    from typing import NotRequired, TypedDict

    class Result(TypedDict):
        name: str
        score: int
        note: NotRequired[str]

    missing: Result = {"name": "小林"}
    wrong: Result = {"name": "小林", "score": "90"}
""", errors=("typeddict-item", "typeddict-item"))
# 分别报告缺少 score 和 score 值类型不匹配。

<string>:8: error: Missing key "score" for TypedDict "Result"  [typeddict-item]
<string>:9: error: Incompatible types (expression has type "str", TypedDict item "score" has type "int")  [typeddict-item]


## 7 Protocol：按接口匹配对象

Protocol 描述对象需要提供的属性或方法。类只要满足接口，就可以被静态检查器接受，不必显式继承该 Protocol；这称为结构化子类型（structural subtyping），也称静态鸭子类型。

下面的 TextReport 没有继承 Renderable，但提供了符合签名的 render()。协议中的省略号表示这里只声明接口。

In [14]:
from typing import Protocol


class Renderable(Protocol):
    """约定可以生成文本的对象。"""

    def render(self) -> str:
        """返回展示文本。"""
        ...


class TextReport:
    """保存一段报告文本。"""

    def __init__(self, text: str) -> None:
        self.text = text

    def render(self) -> str:
        """返回报告文本。"""
        return self.text


def show_report(report: Renderable) -> str:
    """通过约定接口取得文本。"""
    return report.render()


print(show_report(TextReport("完成")))  # 完成

完成


普通 Protocol 不能直接用于 isinstance()。加上 runtime_checkable 装饰器后，可以检查约定成员是否存在，但它不会检查参数签名及属性类型。

因此，运行时协议检查通过不代表方法会返回正确类型；下面对同一个错误接口分别进行运行时与静态观察。

In [15]:
from typing import runtime_checkable


@runtime_checkable
class RuntimeRenderable(Protocol):
    """用于观察运行时协议检查的边界。"""

    def render(self) -> str:
        """约定返回文本。"""
        ...


class NumberReport:
    """提供同名方法，但返回整数。"""

    def render(self) -> int:
        """返回记录数量。"""
        return 3


print(isinstance(NumberReport(), RuntimeRenderable))  # True：只查成员。
# 下面独立做静态检查，对比上面的运行时成员存在性检查。
check_types("""
    from typing import Protocol

    class Renderable(Protocol):
        def render(self) -> str: ...

    class NumberReport:
        def render(self) -> int:
            return 3

    report: Renderable = NumberReport()
""", errors=("assignment",))
# 静态检查进一步指出 render 的返回类型不兼容。

True


<string>:10: error: Incompatible types in assignment (expression has type "NumberReport", variable has type "Renderable")  [assignment]
<string>:10: note: Following member(s) of "NumberReport" have conflicts:
<string>:10: note:     Expected:
<string>:10: note:         def render(self) -> str
<string>:10: note:     Got:
<string>:10: note:         def render(self) -> int


## 8 Self、Final 与 ClassVar

| 名称 | 中文含义／用途 |
| --- | --- |
| Self | 当前类及其子类的自身类型 |
| Final | 不应重新绑定或覆盖的名称 |
| ClassVar | 应通过类设置的类变量 |

返回 self 的方法使用 Self，可以让检查器保留子类类型。若方法始终新建某个固定基类实例，就不能用 Self 承诺返回调用者的子类类型。

Final 不会冻结对象，ClassVar 也不会改变属性查找规则；两者主要表达静态约束。

In [16]:
from typing import ClassVar, Final, Self


class Draft:
    """维护一段可修改的草稿文本。"""

    kind: ClassVar[str] = "文本"

    def __init__(self, text: str) -> None:
        self.text = text

    def rename(self, text: str) -> Self:
        """修改文本并返回自身，以便继续调用。"""
        self.text = text
        return self


class MarkedDraft(Draft):
    """具有额外展示方法的草稿。"""

    def display(self) -> str:
        """返回带标记的文本。"""
        return f"[{self.text}]"


# rename 返回自身，因此子类仍能继续调用自己的 display。
draft = MarkedDraft("初稿").rename("定稿")
print(draft.display(), Draft.kind)  # [定稿] 文本

tags: Final[list[str]] = []
tags.append("已读")
print(tags)  # ['已读']：Final 限制名称重绑定，不限制列表修改。

[定稿] 文本
['已读']


检查器会阻止给 Final 名称重新赋值，以及通过实例重新设置 ClassVar。需要在运行时禁止修改时，应选择实际执行约束的机制。

In [17]:
check_types("""
    from typing import ClassVar, Final

    limit: Final[int] = 3
    limit = 4

    class Draft:
        kind: ClassVar[str] = "文本"

    draft = Draft()
    draft.kind = "图片"
""", errors=("misc", "misc"))
# 两处错误分别来自 Final 与 ClassVar 约束。

<string>:4: error: Cannot assign to final name "limit"  [misc]
<string>:10: error: Cannot assign to class variable "kind" via instance  [misc]


## 9 overload：表达不同调用形式

当不同参数类型对应不同返回类型时，overload 可以分别描述这些关系。它用于静态检查，不根据标注自动选择实现。

普通 .py 文件中，多个 overload 声明后必须有一个同名实现。实现负责运行时分支，签名需要兼容所有声明。

In [18]:
from typing import overload


@overload
def normalize(value: str) -> str: ...


@overload
def normalize(value: None) -> None: ...


def normalize(value: str | None) -> str | None:
    """去除文本两端空白；没有文本时保留 None。"""
    if value is None:
        return None
    return value.strip()


print(normalize(" Python "), normalize(None))  # Python None

Python None


联合类型只说明可能的类型集合，overload 还能说明哪种输入对应哪种输出。调用检查按声明判断，不把更宽的实现签名当作额外的调用形式。

In [19]:
check_types("""
    from typing import overload

    @overload
    def normalize(value: str) -> str: ...
    @overload
    def normalize(value: None) -> None: ...
    def normalize(value: str | None) -> str | None:
        return None if value is None else value.strip()

    reveal_type(normalize(" Python "))
    reveal_type(normalize(None))
""")
# 两条 note 分别显示 str 和 None。

<string>:10: note: Revealed type is "str"
<string>:11: note: Revealed type is "None"


## 10 类型收窄与 TypeGuard

### 10.1 根据实际分支缩小类型范围

类型收窄（type narrowing）指检查器根据条件，把某一位置的类型判断得更具体。常见条件包括 isinstance() 和 is not None；提前 return 后，剩余代码也可以排除已经处理的情况。

收窄依赖实际判断，不会把原对象转换为另一种类型。

In [20]:
check_types("""
    def normalize_input(value: str | int | None) -> str:
        if value is None:
            return "未填写"
        if isinstance(value, int):
            return str(value)
        reveal_type(value)
        return value.strip()
""")
# 最后一个分支中 value 已排除 None 和 int，note 显示 str。

<string>:6: note: Revealed type is "str"


### 10.2 用 TypeGuard 描述自定义判断

TypeGuard[T] 标在判断函数的返回类型上，表示返回 True 时，第一个显式实参符合 T。函数实际返回 bool；T 在这里表示判断成功后承诺的类型。

TypeGuard 只在成功分支提供这种收窄，失败分支不会据此排除 T。检查器信任这个承诺，判断函数必须真实检查条件；共享可变数据被其他代码修改后，已有判断也可能失效。

In [21]:
from typing import TypeGuard


def is_text_pair(
    values: tuple[object, ...],
) -> TypeGuard[tuple[str, str]]:
    """确认对象是恰有两个字符串元素的元组。"""
    return len(values) == 2 and all(
        isinstance(value, str) for value in values
    )


def join_pair(values: tuple[object, ...]) -> str:
    """校验二元文本后拼接；不符合要求时抛出 ValueError。"""
    if not is_text_pair(values):
        raise ValueError("需要两个字符串")
    return " / ".join(values)


print(join_pair(("Python", "mypy")))  # Python / mypy
print(is_text_pair(("Python", 3)), is_text_pair(()))  # False False

Python / mypy
False False


下面把两个分支交给检查器观察。示例使用不可变元组，让注意力集中在“True 承诺了什么”，避免共享列表在检查后被修改。

In [22]:
check_types("""
    from typing import TypeGuard

    def is_text_pair(
        values: tuple[object, ...],
    ) -> TypeGuard[tuple[str, str]]:
        return len(values) == 2 and all(
            isinstance(value, str) for value in values
        )

    def inspect_pair(values: tuple[object, ...]) -> None:
        if is_text_pair(values):
            reveal_type(values)
        else:
            reveal_type(values)
""")
# 成功分支是 tuple[str, str]；失败分支仍是 tuple[object, ...]。

<string>:12: note: Revealed type is "tuple[str, str]"
<string>:14: note: Revealed type is "tuple[object, ...]"


### 10.3 cast() 不做转换或校验

cast(int, value) 只是向检查器声明 value 应按 int 看待，运行时原样返回 value。int(value) 才会尝试转换。

只有掌握了检查器看不到的可靠依据时才使用 cast()；处理未知输入时，优先写出真实判断。

In [23]:
from typing import cast

raw: object = "90"
claimed = cast(int, raw)
converted = int("90")

print(type(claimed).__name__, claimed is raw)  # str True
print(type(converted).__name__, converted)  # int 90

str True
int 90


## 11 前向引用与 TYPE_CHECKING

### 11.1 引用尚未完成定义的类

Python 3.12 通常在执行函数定义时求值其标注。类体内定义方法时，类名本身尚未完成绑定，可以把包含该类名的整个标注写成字符串，这称为前向引用（forward reference）。

只把类名加引号再在外面写联合运算，可能变成字符串与类型的运行时运算；下面把 Node 与 None 的整段标注一起放入引号。

In [24]:
class Node:
    """保存文本和下一个节点的引用。"""

    def __init__(self, text: str, next_node: "Node | None" = None) -> None:
        self.text = text
        self.next_node = next_node


head = Node("起点", Node("终点"))
print(head.next_node.text if head.next_node else "无后继")  # 终点
print(Node.__init__.__annotations__["next_node"])  # Node | None

终点
Node | None


### 11.2 把仅用于标注的导入留给检查器

在 Python 3.12 模块顶部写 from \_\_future\_\_ import annotations，可将函数和变量标注保存为字符串，延后求值。

TYPE_CHECKING 在运行时是 False，检查器按 True 分析其分支。它适合包住仅供标注使用的导入，减少运行时依赖；运行时确实需要的类或函数必须正常导入。需要解析这些字符串的框架仍必须能找到相应名称。

配套 forward_models.py 使用这两个机制：Report 的 date 标注留给检查器，运行时由调用方提供实际日期对象。

In [25]:
from datetime import date
from pathlib import Path
import runpy

scripts = Path("scripts/17-static-typing")
model_namespace = runpy.run_path(str(scripts / "forward_models.py"))
Report = model_namespace["Report"]

report = Report(date(2024, 1, 2))
print(report.created.isoformat())  # 2024-01-02
print(Report.__init__.__annotations__["created"])  # date：保存的是字符串。
print("date" in model_namespace)  # False：仅标注导入没有在运行时执行。

2024-01-02
date
False


## 12 类型声明文件与工程使用

### 12.1 区分实现与 .pyi 声明

.pyi 文件描述模块的公开类型接口，通常用省略号代替函数体。对于同目录下的同名模块，mypy 优先读取 .pyi，Python 运行时执行 .py。

配套 report_value.py 是未标注的整数转换函数，report_value.pyi 声明 parse_count(text: str) -> int。report_client.py 使用该函数；下面分别运行调用方和检查调用方，观察两种工具的职责。

In [26]:
import subprocess
import sys

# 运行调用方时执行 .py；-B 避免留下字节码缓存。
completed = subprocess.run(
    [sys.executable, "-B", str(scripts / "report_client.py")],
    check=True, capture_output=True, text=True, encoding="utf-8",
)
print(completed.stdout.strip())  # 12
assert not completed.stderr, completed.stderr

# 检查调用方时读取 .pyi，并检查上一节仅用于标注的导入。
report, error_report, status = api.run([
    "--python-version", "3.12", "--strict", "--no-color-output",
    "--no-incremental", "--cache-dir", os.devnull,
    str(scripts / "report_client.py"),
    str(scripts / "forward_models.py"),
])
print(report.strip())  # 两个文件通过静态检查。
assert status == 0 and not error_report, (report, error_report)

12


Success: no issues found in 2 source files


### 12.2 第三方库的类型信息从哪里来

| 名称／文件 | 中文含义／用途 |
| --- | --- |
| 内联标注 | 类型信息与库的 Python 实现一起提供 |
| .pyi | 与实现分开的类型声明文件 |
| py.typed | 发布包声明自己提供类型信息的标记文件 |
| stub-only package | 单独分发的类型声明包 |

使用第三方库时，先查它是否自带类型信息；缺少时再选择与库版本匹配的声明包，或为实际使用的接口补充本地 .pyi。这里的配套文件只是本地演示模块，没有发布或安装第三方包。

检查通过依赖代码、声明和配置共同提供的信息。Any、忽略错误的注释或不准确的 .pyi 都可能留下缺口；类型检查不能代替输入校验和行为测试。--strict 开启一组额外规则，具体集合随 mypy 版本变化，本章固定工具版本。

In [27]:
# .pyi 的声明不会替调用方校验文本能否转换为整数。
invalid = subprocess.run(
    [sys.executable, "-B", "-c",
     "from report_value import parse_count; parse_count('many')"],
    cwd=scripts, capture_output=True, text=True, encoding="utf-8",
)
# str 类型正确，但内容不符合 int() 的转换要求。
assert invalid.returncode != 0
assert "ValueError: invalid literal for int()" in invalid.stderr
print(invalid.stderr.strip().splitlines()[-1])  # ValueError: invalid literal for int() with base 10: 'many'。

ValueError: invalid literal for int() with base 10: 'many'


## 本章小结

（1）泛型保留类型之间的对应关系；TypeVar、ParamSpec、TypeVarTuple 分别表达单个类型、函数参数列表和一组类型。

（2）TypedDict 描述字段，Protocol 描述对象接口；它们都不代替运行时数据校验。

（3）Self、Final、ClassVar 和 overload 让接口约定更精确。类型收窄依赖分支判断，cast() 不会转换对象。

（4）前向引用与 TYPE_CHECKING 处理标注的求值和导入时机；mypy 与 .pyi 提供静态检查，执行测试继续负责真实行为。

理解自查：能否分别解释“能运行但类型检查报错”和“类型检查通过但运行失败”的原因？

## 练习

### 练习 1：选择读取接口

下面两条调用，哪一条会被 mypy 接受？先预测，再运行。说明 list 的不变性为什么与 append() 有关。

In [28]:
# 直接保留诊断，运行后对照自己的预测；这里不提供答案断言。
exercise_report, exercise_error, exercise_status = api.run([
    "--python-version", "3.12", "--strict", "--no-color-output",
    "--no-incremental", "--cache-dir", os.devnull,
    "-c", dedent("""
        from collections.abc import Sequence

        def view(values: Sequence[object]) -> int:
            return len(values)

        def add(values: list[object]) -> None:
            values.append(1)

        words: list[str] = ["a"]
        view(words)
        add(words)
    """).strip(),
])
print(exercise_report.strip())
assert not exercise_error, exercise_error
assert exercise_status in (0, 1), exercise_status

<string>:11: error: Argument 1 to "add" has incompatible type "list[str]"; expected "list[object]"  [arg-type]
<string>:11: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
<string>:11: note: Consider using "Sequence" instead, which is covariant
Found 1 error in 1 file (checked 1 source file)


### 练习 2：保留类型关系

编写 last[T](items: Sequence[T]) -> T，返回非空序列的末项。

分别用整数列表和字符串元组调用，检查运行结果，并用 check_types() 中的 reveal_type() 确认返回类型。空序列应明确抛出 IndexError，不返回含义模糊的默认值。

In [29]:
# 在这里编写 last()、调用示例和独立的静态检查源码。

### 练习 3：检查接口与输入

定义 HasName 协议，约定只读的 name 属性为 str；让两个不同的类满足它，编写接收 HasName 的 greet()。

再定义一个错误类，使 name 返回 int。正确调用应通过 mypy，错误调用应被拒绝；说明为什么不能只用 runtime_checkable 判断属性的返回类型。提示：只读属性可以用 property 声明。

In [30]:
# 在这里编写协议、两个实现和静态检查示例。

## 参考与引用来源

| 网站 | 资料与知识点定位 |
| --- | --- |
| Python 官方文档 | Python 3.12：[类型参数语法](https://docs.python.org/3.12/reference/compound_stmts.html#type-params)；typing 的[泛型](https://docs.python.org/3.12/library/typing.html#generics)、[TypeVar](https://docs.python.org/3.12/library/typing.html#typing.TypeVar)、[可调用对象标注](https://docs.python.org/3.12/library/typing.html#annotating-callable-objects)、[生成器标注](https://docs.python.org/3.12/library/typing.html#annotating-generators-and-coroutines)、[ParamSpec](https://docs.python.org/3.12/library/typing.html#typing.ParamSpec)、[TypeVarTuple](https://docs.python.org/3.12/library/typing.html#typing.TypeVarTuple)、[TypedDict](https://docs.python.org/3.12/library/typing.html#typing.TypedDict)、[Protocol](https://docs.python.org/3.12/library/typing.html#typing.Protocol)、[runtime_checkable](https://docs.python.org/3.12/library/typing.html#typing.runtime_checkable)、[Self](https://docs.python.org/3.12/library/typing.html#typing.Self)、[Final](https://docs.python.org/3.12/library/typing.html#typing.Final)、[ClassVar](https://docs.python.org/3.12/library/typing.html#typing.ClassVar)、[overload](https://docs.python.org/3.12/library/typing.html#typing.overload)、[TypeGuard](https://docs.python.org/3.12/library/typing.html#typing.TypeGuard)、[cast](https://docs.python.org/3.12/library/typing.html#typing.cast)、[TYPE_CHECKING](https://docs.python.org/3.12/library/typing.html#typing.TYPE_CHECKING)、[标注解析边界](https://docs.python.org/3.12/library/typing.html#typing.get_type_hints)；[runpy.run_path](https://docs.python.org/3.12/library/runpy.html#runpy.run_path)、[subprocess.run](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)。 |
| mypy 文档 | mypy 2.3.1：[API 调用](https://mypy.readthedocs.io/en/stable/extending_mypy.html#integrating-mypy-into-another-python-application)、[命令行与严格检查](https://mypy.readthedocs.io/en/stable/command_line.html#cmdoption-mypy-strict)、[缓存设置](https://mypy.readthedocs.io/en/stable/command_line.html#cmdoption-mypy-cache-dir)、[泛型函数](https://mypy.readthedocs.io/en/stable/generics.html#generic-functions)、[容器变型规则](https://mypy.readthedocs.io/en/stable/generics.html#variance-of-generic-types)、[类型收窄](https://mypy.readthedocs.io/en/stable/type_narrowing.html#type-narrowing-expressions)、[TypeGuard](https://mypy.readthedocs.io/en/stable/type_narrowing.html#user-defined-type-guards)、[类型声明文件](https://mypy.readthedocs.io/en/stable/stubs.html#creating-a-stub)、[第三方包类型信息](https://mypy.readthedocs.io/en/stable/installed_packages.html#using-installed-packages-with-mypy-pep-561)。 |